In [1]:
# --- الخلية 1: التثبيت الذكي (Smart Offline Install) ---
import sys
import os
import glob
import subprocess

print("⚙️ جاري تثبيت المكتبات الضرورية فقط (حماية الـ GPU)...")

# 1. تحديد المكتبات الخطرة التي لا يجب إعادة تثبيتها لأنها موجودة ومجهزة للـ GPU
FORBIDDEN_PACKAGES = [
    "torch-", "torchvision", "torchaudio",  # نترك نسخ Kaggle الأصلية
    "numpy", "pandas", "opencv", "matplotlib", "scipy", "pillow" # موجودة مسبقاً
]

# 2. العثور على الملفات
all_whls = glob.glob("/kaggle/input/**/*.whl", recursive=True)
print(f"📦 تم العثور على {len(all_whls)} ملف في الداتا سيت.")

success_count = 0
for path in all_whls:
    filename = os.path.basename(path).lower()
    
    # التحقق: هل هذا الملف "ممنوع"؟
    is_forbidden = False
    for forbidden in FORBIDDEN_PACKAGES:
        if forbidden in filename:
            is_forbidden = True
            break
    
    if is_forbidden:
        # print(f"🛡️ تم الحفاظ على نسخة النظام من: {filename}")
        continue

    # التثبيت الآمن للمكتبات المفقودة فقط (مثل YOLO, SMP)
    try:
        subprocess.check_call([
            sys.executable, "-m", "pip", "install", 
            path, 
            "--no-deps",
            "--ignore-installed"
        ], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
        success_count += 1
        # print(f"✅ تم تثبيت: {filename}")
    except Exception as e:
        print(f"⚠️ فشل تثبيت: {filename}")

print(f"✅ تمت العملية! تم تثبيت {success_count} مكتبة إضافية.")
print("⚡ إعدادات الـ Torch الحالية:")
import torch
print(f"   - Torch Version: {torch.__version__}")
print(f"   - CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"   - GPU Name: {torch.cuda.get_device_name(0)}")
else:
    print("❌ تنبيه خطير: الـ GPU غير مفعل! تأكد من إعدادات Kaggle Sidebar.")

# إضافة مسارات YOLO إذا لزم الأمر
src_dirs = glob.glob("/kaggle/input/**/ultralytics", recursive=True)
for d in src_dirs:
    parent = os.path.dirname(d)
    if parent not in sys.path:
        sys.path.append(parent)

⚙️ جاري تثبيت المكتبات الضرورية فقط (حماية الـ GPU)...
📦 تم العثور على 95 ملف في الداتا سيت.
⚠️ فشل تثبيت: triton-3.0.0-1-cp310-cp310-manylinux2014_x86_64.manylinux_2_17_x86_64.whl
⚠️ فشل تثبيت: pyyaml-6.0.2-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl
⚠️ فشل تثبيت: kiwisolver-1.4.5-cp310-cp310-manylinux_2_12_x86_64.manylinux2010_x86_64.whl
⚠️ فشل تثبيت: markupsafe-2.1.5-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl
⚠️ فشل تثبيت: fonttools-4.53.1-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl
⚠️ فشل تثبيت: charset_normalizer-3.3.2-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl
⚠️ فشل تثبيت: contourpy-1.2.1-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl
⚠️ فشل تثبيت: pyyaml-6.0.3-cp310-cp310-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl
⚠️ فشل تثبيت: triton-3.5.1-cp310-cp310-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl
⚠️ فشل تثبيت: markupsafe-3.0.3-cp310-cp310-manylinux2014_x86_64.manylinux_2_17_x8

In [2]:
# --- Cell 2: Offline install/import fix (NO internet) ---
import sys, os, glob, subprocess, site
from importlib import reload

print("🔧 Cell 2: Offline install/import fix (no internet).")

# تعطيل أي شيء ممكن يحاول يتصل بالنت
os.environ["WANDB_DISABLED"] = "true"
os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

def ensure_pkg(import_name: str, wheel_hint: str):
    """Try import; if missing install from /kaggle/input wheels offline."""
    try:
        __import__(import_name)
        print(f"✅ Already available: {import_name}")
        return True
    except Exception:
        wheels = glob.glob(f"/kaggle/input/**/*{wheel_hint}*.whl", recursive=True)
        if not wheels:
            print(f"❌ Missing wheel for: {import_name} (hint={wheel_hint})")
            return False
        target = sorted(wheels, key=len)[0]
        try:
            subprocess.check_call(
                [sys.executable, "-m", "pip", "install", target, "--no-deps", "--ignore-installed", "--quiet"],
                stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL
            )
            reload(site)
            __import__(import_name)
            print(f"✅ Installed offline: {import_name}")
            return True
        except Exception as e:
            print(f"⚠️ Offline install failed for {import_name}: {e}")
            return False

# نضمن فقط المكتبات التي نحتاجها (بدون لمس torch)
ensure_pkg("timm", "timm")
ensure_pkg("segmentation_models_pytorch", "segmentation_models_pytorch")
ensure_pkg("ultralytics", "ultralytics")

# محرك parquet (عادة موجود في Kaggle)
try:
    import pyarrow  # noqa
    print("✅ pyarrow available (parquet OK)")
except Exception:
    # إذا مو موجود غالبًا Kaggle يوفره؛ لا نحاول تنزيله من النت
    print("⚠️ pyarrow not found. If parquet read fails later, Kaggle environment may be missing parquet engine.")

import torch
import segmentation_models_pytorch as smp

print("------ Environment Check ------")
print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))
print("smp:", getattr(smp, "__version__", "unknown"))
print("✅ Cell 2 ready.")


🔧 Cell 2: Offline install/import fix (no internet).


/usr/local/lib/python3.11/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'frozen' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'frozen' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` 

✅ Already available: timm
✅ Installed offline: segmentation_models_pytorch
✅ Already available: ultralytics
✅ pyarrow available (parquet OK)
------ Environment Check ------
torch: 2.6.0+cu124
cuda available: True
gpu: Tesla T4
smp: 0.5.0
✅ Cell 2 ready.


In [3]:
# --- الخلية 3: محرك الرسم الحي (Ultimate Renderer) - [M3: UPDATED] ---
# إعداد قاعدة البيانات (تحميل مرة واحدة فقط)
DB_DIR = "physionet_db"
if not os.path.exists(DB_DIR):
    os.makedirs(DB_DIR)
    print("⬇️ جاري تحميل سجلات PTB-XL الأساسية...")
    try:
        # تحميل عينة كافية لضمان التنوع
        records = wfdb.get_record_list('ptb-xl')
        selected_records = records[:200] 
        wfdb.dl_database('ptb-xl', os.getcwd() + "/" + DB_DIR, selected_records)
        print(f"✅ تم تحميل {len(selected_records)} سجل.")
    except Exception as e:
        print(f"⚠️ تحذير: فشل التحميل ({e})، سيتم استخدام التوليد الاحتياطي.")

class UltimateRenderer:
    def __init__(self, db_dir):
        self.db_dir = db_dir
        self.records = [f.split('.')[0] for f in os.listdir(db_dir) if f.endswith('.dat')] if os.path.exists(db_dir) else []
        
    def get_real_signal(self):
        """سحب إشارة عشوائية من PTB-XL"""
        if not self.records:
            t = np.linspace(0, 4, 2000); return np.sin(2 * np.pi * 5 * t) # Fallback
            
        try:
            rec_name = random.choice(self.records)
            record, meta = wfdb.rdsamp(f"{self.db_dir}/{rec_name}")
            lead_idx = random.randint(0, record.shape[1] - 1)
            signal = record[:, lead_idx]
            signal = np.nan_to_num(signal)
            
            # قص عشوائي (Zoom in/out simulation)
            crop_len = random.randint(1000, 3000)
            if len(signal) > crop_len:
                start = random.randint(0, len(signal) - crop_len)
                return signal[start : start+crop_len]
            return signal
        except:
            return np.random.randn(2000)

    def render_to_memory(self, signal):
        """الرسم بدقة عالية (DPI=200) في الذاكرة مباشرة"""
        # 3. الشبكة المتغيرة (Distractor)
        grid_color = random.choice(['red', '#ffb6c1', '#cfcfcf', '#e0e0e0', 'pink'])
        grid_alpha = random.uniform(0.3, 0.8)
        bg_color = random.choice(['white', '#fffdf5', '#f0f0f0']) 
        
        fig_h, fig_w = 3, 8
        dpi = 200 # حسب الطلب لضمان وضوح الحواف
        
        # --- أ. توليد القناع (Mask Generation) ---
        fig, ax = plt.subplots(figsize=(fig_w, fig_h), dpi=dpi)
        ax.plot(signal, color='white', linewidth=3.0) 
        ax.set_ylim(np.min(signal), np.max(signal))
        ax.axis('off')
        fig.patch.set_facecolor('black')
        
        fig.canvas.draw()
        mask = np.frombuffer(fig.canvas.tostring_rgb(), dtype=np.uint8)
        mask = mask.reshape(fig.canvas.get_width_height()[::-1] + (3,))
        mask = cv2.cvtColor(mask, cv2.COLOR_RGB2GRAY)
        _, mask = cv2.threshold(mask, 10, 255, cv2.THRESH_BINARY)
        plt.close(fig)

        # --- ب. الرسم الأولي (Rendering) ---
        fig, ax = plt.subplots(figsize=(fig_w, fig_h), dpi=dpi)
        ax.set_facecolor(bg_color)
        
        # رسم الشبكة
        ax.minorticks_on()
        ax.grid(which='major', linestyle='-', linewidth=0.8, color=grid_color, alpha=grid_alpha)
        ax.grid(which='minor', linestyle=':', linewidth=0.4, color=grid_color, alpha=grid_alpha*0.5)
        
        # رسم الإشارة (محاكاة ألوان الحبر المختلفة)
        line_color = random.choice(['black', 'blue', '#000033']) 
        ax.plot(signal, color=line_color, linewidth=random.uniform(1.0, 1.8))
        
        ax.axis('off')
        ax.set_ylim(np.min(signal), np.max(signal))
        fig.patch.set_facecolor(bg_color)
        
        fig.canvas.draw()
        img = np.frombuffer(fig.canvas.tostring_rgb(), dtype=np.uint8)
        img = img.reshape(fig.canvas.get_width_height()[::-1] + (3,))
        # نحتفظ بها BGR هنا لأن OpenCV يفضل ذلك للمعالجة اللاحقة (Distractors)
        img = cv2.cvtColor(img, cv2.COLOR_RGB2BGR) 
        plt.close(fig)
        
        return img, mask

print("✅ تم تحديث الخلية 3: محرك الرسم الحي جاهز (DPI=200).")

⬇️ جاري تحميل سجلات PTB-XL الأساسية...
⚠️ تحذير: فشل التحميل (name 'wfdb' is not defined)، سيتم استخدام التوليد الاحتياطي.
✅ تم تحديث الخلية 3: محرك الرسم الحي جاهز (DPI=200).


In [4]:
# --- Cell 22: V47 Benchmarking Engine (Based on Winner V46) ---
# [MODIFIABLE MODEL SELECTOR - SAME ROBUST LOGIC]

import gc, os, csv, glob, math, re
import numpy as np
import pandas as pd
import cv2
import torch
from collections import OrderedDict
from tqdm import tqdm
from scipy.signal import savgol_filter, resample, find_peaks, butter, filtfilt
import segmentation_models_pytorch as smp
from ultralytics import YOLO

# ============================================================
# 🎛️ CONTROL PANEL (لوحة التحكم بالموديلات)
# ============================================================
# غير القيمة إلى True للموديل الذي تريد اختباره، و False للباقي
USE_RESNET       = False   # الموديل الأساسي (Phase 10)
USE_EFFICIENTNET = False  # المنافس الأول
USE_DEEPLAB      = True  # المنافس الثاني
# ============================================================

# ---------------------------
# 0) Constants & Config
# ---------------------------
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
TEST_CSV_PATH = "/kaggle/input/physionet-ecg-image-digitization/test.csv"
SAMPLE_PARQUET_PATH = "/kaggle/input/physionet-ecg-image-digitization/sample_submission.parquet"
SUBMISSION_FILE = "submission.csv"

# نفس مسارات البحث الشاملة التي نجحت في V46
IMAGE_DIRS = [
    "/kaggle/input/physionet-ecg-image-digitization",
    "/kaggle/input"
]

MODELS_DIR = "/kaggle/input/ecg-final-models1"

LEAD_NAMES = ['I', 'II', 'III', 'aVR', 'aVL', 'aVF', 'V1', 'V2', 'V3', 'V4', 'V5', 'V6']
LEAD_TO_IDX = {name: i for i, name in enumerate(LEAD_NAMES)}
TARGET_H = 256
YOLO_INF_MAX = 1280
YOLO_CONF = 0.10
MAX_W = 2048
MAX_CACHE = 15
DP_MAX_W = 1400
DP_SMOOTH = 0.45

print(f"⚡ Device: {DEVICE}")
print(f"🧪 Testing Configuration: ResNet={USE_RESNET} | EffNet={USE_EFFICIENTNET} | DeepLab={USE_DEEPLAB}")

# ---------------------------
# 1) Robust ID Helper (The Winner Logic)
# ---------------------------
def normalize_id(text):
    """استخراج الأرقام فقط لضمان العثور على الصور"""
    return "".join(filter(str.isdigit, str(text)))

# ---------------------------
# 2) Read template + Dynamic lengths
# ---------------------------
if not os.path.exists(SAMPLE_PARQUET_PATH):
    try:
        sample_df = pd.read_csv(SAMPLE_PARQUET_PATH.replace('.parquet', '.csv'))
        template_ids = sample_df["id"].astype(str).to_numpy()
    except:
        template_ids = []
        print("⚠️ Template not found.")
else:
    print("📦 Reading Parquet template ids...")
    template = pd.read_parquet(SAMPLE_PARQUET_PATH, columns=["id"])
    template_ids = template["id"].astype(str).to_numpy()
    del template
    gc.collect()

print(f"✅ Template rows: {len(template_ids):,}")

# Dynamic length map
patient_lengths = {}
print("🧠 Scanning template for patient lengths...")
for rid in tqdm(template_ids, desc="Scanning Lengths"):
    try:
        parts = rid.rsplit("_", 2)
        if len(parts) != 3: continue
        pid = str(parts[0])
        idx = int(parts[1])
        k = normalize_id(pid)
        cur = patient_lengths.get(k, 0)
        if idx + 1 > cur: patient_lengths[k] = idx + 1
    except: pass
print(f"✅ Lengths mapped for {len(patient_lengths):,} patients.")

# ---------------------------
# 3) Index images (Robust Mode)
# ---------------------------
print("🗂️ Indexing images (Robust Mode)...")
image_paths = []
for d in IMAGE_DIRS:
    image_paths += glob.glob(f"{d}/**/*.png", recursive=True)
    image_paths += glob.glob(f"{d}/**/*.jpg", recursive=True)

path_map = {}
for p in image_paths:
    fname = os.path.basename(p)
    k = normalize_id(fname)
    if k: path_map[k] = p

print(f"✅ Indexed images: {len(path_map):,} (Unique numeric IDs)")

fs_map = {}
if os.path.exists(TEST_CSV_PATH):
    try:
        tdf = pd.read_csv(TEST_CSV_PATH, dtype=str)
        cols_low = {c.lower(): c for c in tdf.columns}
        id_c = next((cols_low[c] for c in cols_low if "id" in c), None)
        fs_c = next((cols_low[c] for c in cols_low if "fs" in c), None)
        if id_c and fs_c:
            for _, row in tdf.iterrows():
                fs_map[normalize_id(row[id_c])] = row[fs_c]
        print(f"✅ fs_map loaded: {len(fs_map):,} items")
    except: pass

# ---------------------------
# 4) Load Models (BASED ON CONFIG)
# ---------------------------
print("⚙️ Loading models based on configuration...")

CLASS_TO_LEAD_IDX = {}
yolo_model = None
YOLO_PATH = f"{MODELS_DIR}/best.pt"

# Load YOLO (Always needed)
if os.path.exists(YOLO_PATH):
    try:
        yolo_model = YOLO(YOLO_PATH)
        print(f"✅ YOLO loaded.")
        if yolo_model.names:
            names = yolo_model.names
            items = list(names.items()) if isinstance(names, dict) else list(enumerate(names))
            for cid, cname in items:
                s = str(cname).strip().replace("Lead", "").replace("_", "").replace(" ", "")
                s = s.replace("AVR", "aVR").replace("AVL", "aVL").replace("AVF", "aVF")
                s = s.replace("AVr","aVR").replace("AVl","aVL").replace("AVf","aVF")
                if s in LEAD_TO_IDX:
                    CLASS_TO_LEAD_IDX[int(cid)] = LEAD_TO_IDX[s]
            for i in range(12): CLASS_TO_LEAD_IDX.setdefault(i, i)
    except: print("⚠️ YOLO load failed.")
else:
    print("⚠️ YOLO not found.")

# --- Load Selected U-Nets ---
unet_models = []

# 1. ResNet50
if USE_RESNET:
    p = f"{MODELS_DIR}/best_model_phase10.pth"
    if os.path.exists(p):
        try:
            print(f"🔹 Loading ResNet50...")
            m = smp.Unet(encoder_name="resnet50", encoder_weights=None, in_channels=3, classes=1, decoder_attention_type="scse")
            m.load_state_dict(torch.load(p, map_location=DEVICE))
            unet_models.append(m.to(DEVICE).eval())
            print(f"✅ ResNet50 Loaded.")
        except Exception as e: print(f"❌ ResNet50 Error: {e}")
    else: print(f"⚠️ ResNet50 file missing: {p}")

# 2. EfficientNet-B3
if USE_EFFICIENTNET:
    p = f"{MODELS_DIR}/best_model_efficientnet_ph10.pth"
    if os.path.exists(p):
        try:
            print(f"🔹 Loading EfficientNet-B3...")
            m = smp.Unet(encoder_name="efficientnet-b3", encoder_weights=None, in_channels=3, classes=1, decoder_attention_type="scse")
            m.load_state_dict(torch.load(p, map_location=DEVICE))
            unet_models.append(m.to(DEVICE).eval())
            print(f"✅ EfficientNet-B3 Loaded.")
        except Exception as e: print(f"❌ EfficientNet Error: {e}")
    else: print(f"⚠️ EfficientNet file missing: {p}")

# 3. DeepLabV3+
if USE_DEEPLAB:
    p = f"{MODELS_DIR}/best_model_deeplab_ph10.pth"
    if os.path.exists(p):
        try:
            print(f"🔹 Loading DeepLabV3+...")
            m = smp.DeepLabV3Plus(encoder_name="resnet50", encoder_weights=None, in_channels=3, classes=1)
            m.load_state_dict(torch.load(p, map_location=DEVICE))
            unet_models.append(m.to(DEVICE).eval())
            print(f"✅ DeepLabV3+ Loaded.")
        except Exception as e: print(f"❌ DeepLab Error: {e}")
    else: print(f"⚠️ DeepLab file missing: {p}")

if not unet_models:
    print("⚠️ WARNING: No models selected! Submission will contain zeros.")

# ---------------------------
# 5) Core Functions (V46 Logic - Safe & Robust)
# ---------------------------

def get_image_path_robust(pid):
    return path_map.get(normalize_id(pid))

def safe_read_rgb(path):
    try:
        img = cv2.imread(path)
        return cv2.cvtColor(img, cv2.COLOR_BGR2RGB) if img is not None else None
    except: return None

def preprocess_remove_grid_rgb(img_rgb):
    try:
        hsv = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2HSV)
        mask = (cv2.inRange(hsv, (0, 50, 50), (10, 255, 255)) |
                cv2.inRange(hsv, (170, 50, 50), (180, 255, 255)))
        out = img_rgb.copy()
        out[mask > 0] = (255, 255, 255)
        return out
    except: return img_rgb

def apply_high_pass(data, cutoff=0.5, fs=500.0, order=3):
    try:
        if len(data) < order * 3: return data
        nyq = 0.5 * fs
        wn = cutoff / nyq
        if not (0 < wn < 1): return data
        b, a = butter(order, wn, btype="high")
        return filtfilt(b, a, data)
    except: return data

def viterbi_adaptive(prob_map):
    H, W = prob_map.shape
    if W < 5: return np.zeros(W)
    path = np.argmax(prob_map, axis=0).astype(np.float32)
    win = 11 if W >= 11 else (W if W % 2 == 1 else W-1)
    if win > 3:
        try: path = savgol_filter(path, win, 2)
        except: pass
    return (H - path).astype(np.float32)

def predict_ensemble(crops_rgb):
    if not unet_models: return []
    
    tens, scales, widths = [], [], []
    for c in crops_rgb:
        h, w = c.shape[:2]
        sc = TARGET_H / float(h)
        nw = int(w * sc)
        nw = min(nw, MAX_W)
        rz = cv2.resize(c, (nw, TARGET_H))
        t = torch.from_numpy(rz).permute(2, 0, 1).float() / 255.0
        tens.append(t)
        scales.append(sc)
        widths.append(nw)
        
    if not tens: return []
    max_w = int(math.ceil(max(widths) / 32) * 32)
    batch = torch.zeros((len(tens), 3, TARGET_H, max_w), device=DEVICE)
    for i, t in enumerate(tens):
        batch[i, :, :, :t.shape[2]] = t.to(DEVICE)
        
    final_preds = None
    with torch.no_grad():
        with torch.amp.autocast('cuda', enabled=(DEVICE=="cuda")):
            for model in unet_models:
                out = torch.sigmoid(model(batch))
                if final_preds is None: final_preds = out
                else: final_preds += out
    
    final_preds /= len(unet_models)
    final_preds = final_preds.cpu().numpy()
    
    results = []
    for i in range(len(tens)):
        w_real = widths[i]
        pmap = final_preds[i, 0, :, :w_real]
        results.append((pmap, scales[i]))
        
    return results

def get_yolo_crops_mapped(img):
    crops = [None]*12
    if yolo_model is None: return crops
    try:
        h, w = img.shape[:2]
        scale = 1.0
        if max(h, w) > YOLO_INF_MAX:
            scale = YOLO_INF_MAX / max(h, w)
            img_in = cv2.resize(img, (0,0), fx=scale, fy=scale)
        else:
            img_in = img
            
        res = yolo_model.predict(img_in, conf=YOLO_CONF, verbose=False, device=DEVICE)
        if not res: return crops
        
        boxes = res[0].boxes.data.cpu().numpy()
        best_leads = {}
        for b in boxes:
            x1, y1, x2, y2, conf, cls = b
            l_idx = CLASS_TO_LEAD_IDX.get(int(cls), int(cls))
            if l_idx < 0 or l_idx > 11: continue
            x1, x2 = x1/scale, x2/scale
            y1, y2 = y1/scale, y2/scale
            if l_idx not in best_leads or conf > best_leads[l_idx][0]:
                best_leads[l_idx] = (conf, [int(x1), int(y1), int(x2), int(y2)])
                
        for l_idx, (_, coords) in best_leads.items():
            x1, y1, x2, y2 = coords
            x1, y1 = max(0, x1), max(0, y1)
            x2, y2 = min(w, x2), min(h, y2)
            if x2 > x1 and y2 > y1:
                crops[l_idx] = img[y1:y2, x1:x2]
    except: pass
    return crops

# ---------------------------
# 6) Main Processing Loop (V46 - SAFE SCALING)
# ---------------------------
patient_cache = OrderedDict()

def process_patient(pid, t_len):
    path = get_image_path_robust(pid)
    if not path: return np.zeros((12, t_len))
    
    img = safe_read_rgb(path)
    if img is None: return np.zeros((12, t_len))
    
    crops = get_yolo_crops_mapped(img)
    valid_indices, valid_crops_rgb = [], []
    
    for i, c in enumerate(crops):
        if c is not None and c.size > 0:
            valid_indices.append(i)
            valid_crops_rgb.append(preprocess_remove_grid_rgb(c))
            
    if not valid_indices: return np.zeros((12, t_len))
    
    results = predict_ensemble(valid_crops_rgb)
    out_leads = np.zeros((12, t_len))
    fs = float(fs_map.get(normalize_id(pid), 500.0))
    
    for k, (pmap, sc) in enumerate(results):
        l_idx = valid_indices[k]
        raw = viterbi_adaptive(pmap)
        
        # --- SAFE SCALING (V46 Strategy) ---
        # استخدام قيمة ثابتة آمنة (250.0) بدلاً من المعايرة المعقدة
        # لتجنب القيم الشاذة التي دمرت السكور في V45
        scale_factor = 250.0 
        
        mv = (raw - np.median(raw)) / scale_factor
        mv = np.nan_to_num(mv)
        mv = apply_high_pass(mv, fs=fs)
        
        if len(mv) > 0:
            out_leads[l_idx] = resample(mv, t_len)
            
    return out_leads

print("🚀 Starting Submission Loop (BENCHMARK MODE)...")
with open(SUBMISSION_FILE, "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["id", "value"])
    
    debug_counter = 0
    
    for rid in tqdm(template_ids, desc="Processing"):
        try:
            parts = rid.rsplit("_", 2)
            if len(parts) != 3: 
                writer.writerow([rid, "0.0000"])
                continue
            
            pid_raw, idx_raw, lname = parts
            pid_clean = normalize_id(pid_raw)
            idx = int(idx_raw)
            
            if pid_clean in patient_cache:
                leads_data = patient_cache[pid_clean]
                patient_cache.move_to_end(pid_clean)
            else:
                t_len = patient_lengths.get(pid_clean, 5000)
                leads_data = process_patient(pid_raw, t_len)
                
                if debug_counter < 3 and np.any(leads_data):
                    print(f"✅ Processed {pid_raw}: Found signal!")
                    debug_counter += 1
                
                patient_cache[pid_clean] = leads_data
                if len(patient_cache) > MAX_CACHE:
                    patient_cache.popitem(last=False)
            
            l_idx = LEAD_TO_IDX.get(lname, 0)
            val = 0.0
            if l_idx < 12 and idx < leads_data.shape[1]:
                val = leads_data[l_idx, idx]
            
            writer.writerow([rid, f"{val:.4f}"])
            
        except Exception as e:
            writer.writerow([rid, "0.0000"])

print("✅ Submission file generated successfully.")

⚡ Device: cuda
🧪 Testing Configuration: ResNet=False | EffNet=False | DeepLab=True
📦 Reading Parquet template ids...
✅ Template rows: 75,000
🧠 Scanning template for patient lengths...


Scanning Lengths: 100%|██████████| 75000/75000 [00:00<00:00, 605977.85it/s]

✅ Lengths mapped for 2 patients.
🗂️ Indexing images (Robust Mode)...


✅ Indexed images: 8,795 (Unique numeric IDs)
✅ fs_map loaded: 2 items
⚙️ Loading models based on configuration...
✅ YOLO loaded.
🔹 Loading DeepLabV3+...
✅ DeepLabV3+ Loaded.
🚀 Starting Submission Loop (BENCHMARK MODE)...


Processing:  34%|███▍      | 25694/75000 [00:03<00:04, 11695.14it/s]

✅ Processed 1053922973: Found signal!


Processing: 100%|██████████| 75000/75000 [00:04<00:00, 16890.20it/s]

✅ Processed 2352854581: Found signal!
✅ Submission file generated successfully.


In [5]:
# =========================
# --- الخلية 23: Strict Submission Validator (Must Pass) ---
# =========================
import os
import numpy as np
import pandas as pd

SAMPLE_PARQUET_PATH = "/kaggle/input/physionet-ecg-image-digitization/sample_submission.parquet"
SUBMISSION_FILE = "submission.csv"

print("🔍 Validating submission.csv strictly...")

if not os.path.exists(SUBMISSION_FILE):
    raise FileNotFoundError("❌ submission.csv not found. Run Cell 22 first.")

# اقرأ القالب (ids فقط)
tmpl = pd.read_parquet(SAMPLE_PARQUET_PATH, columns=["id"])
tmpl_ids = tmpl["id"].astype(str).to_numpy()
n_tmpl = len(tmpl_ids)

# اقرأ ملفك
sub = pd.read_csv(SUBMISSION_FILE)

# 1) الأعمدة
assert list(sub.columns) == ["id", "value"], f"❌ Columns mismatch: {sub.columns}"

# 2) عدد الصفوف
assert len(sub) == n_tmpl, f"❌ Row count mismatch: sub={len(sub)} vs template={n_tmpl}"

# 3) عدم وجود NaN
assert sub["id"].isna().sum() == 0, "❌ Found NaN in id"
assert sub["value"].isna().sum() == 0, "❌ Found NaN in value"

# 4) تحويل value إلى float والتأكد finite
vals = pd.to_numeric(sub["value"], errors="coerce").to_numpy()
assert np.isfinite(vals).all(), "❌ Found non-finite values (NaN/inf) in value"

# 5) مطابقة أول وآخر ID (كفاية لاكتشاف تغيير ترتيب/تنظيف خاطئ)
sub_ids = sub["id"].astype(str).to_numpy()
assert sub_ids[0] == tmpl_ids[0], f"❌ First ID mismatch: {sub_ids[0]} != {tmpl_ids[0]}"
assert sub_ids[-1] == tmpl_ids[-1], f"❌ Last ID mismatch: {sub_ids[-1]} != {tmpl_ids[-1]}"

# 6) فحص عينة عشوائية للمطابقة (بدون مقارنة كل شيء لتوفير وقت)
idxs = np.linspace(0, n_tmpl-1, num=min(20, n_tmpl), dtype=int)
for i in idxs:
    if sub_ids[i] != tmpl_ids[i]:
        raise AssertionError(f"❌ ID mismatch at row {i}: {sub_ids[i]} != {tmpl_ids[i]}")

size_mb = os.path.getsize(SUBMISSION_FILE) / (1024*1024)
print("✅ All checks passed.")
print(f"📦 submission.csv size: {size_mb:.2f} MB")
print("🎉 Ready to Submit.")


🔍 Validating submission.csv strictly...
✅ All checks passed.
📦 submission.csv size: 1.94 MB
🎉 Ready to Submit.
